<a href="https://colab.research.google.com/github/meryuzlu/QSVT/blob/main/Manual_QSVT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spectral Transformations via Manual Block Encoding, Manual QSVT, and QPE Spectroscopy

The goal of this project is to build the mathematical and algorithmic foundations of Quantum Singular Value Transformation (QSVT) from the ground up rather than treating it as a black-box routine. Starting from a simple two-qubit (2-local) Hamiltonian, we first analyze its spectrum using classical linear algebra and verify polynomial functional calculus through the spectral theorem. We then manually construct a block encoding of the Hamiltonian, manually assemble the QSVT operator from the block encoding and projector-controlled phase operators, and verify that it performs the desired polynomial transformation. Finally, we perform Quantum Phase Estimation (QPE) spectroscopy on the same Hamiltonian and compare the estimated energy spectrum with the exact eigenvalues.

Throughout the notebook, every major component of the QSVT pipeline is implemented explicitly. The only PennyLane routine used in the QSVT implementation is the classical preprocessing function that computes the QSP phase angles from a target polynomial.



## Project Roadmap

The notebook is organized as follows.

1. **Construct a simple 2-local Hamiltonian.**
2. **Analyze its spectrum and verify polynomial functional calculus using the spectral theorem.**
3. **Manually construct a block encoding of the Hamiltonian.**
4. **Manually assemble the QSVT operator from the block encoding and projector-controlled phase operators.**
5. **Apply several polynomial spectral transformations and verify the results.**
6. **Perform Quantum Phase Estimation (QPE) spectroscopy on the same Hamiltonian.**
7. **Estimate the spectrum from both exact eigenstates and a non-eigenstate input.**




In [ ]:
!pip install pennylane
!pip install pennylane scipy

import numpy as np
import pennylane as qml
from scipy.linalg import expm
import pandas as pd


np.set_printoptions(precision=6, suppress=True) # Display NumPy arrays with cleaner formatting.

## Part I — Constructing a Simple 2-Local Hamiltonian

We begin by constructing the Hamiltonian that will be used throughout this notebook. The operator is built from tensor products of the Pauli operators and serves as the central example for every subsequent section of the project.

Although the Hamiltonian is intentionally small, it contains all of the essential mathematical ingredients needed to illustrate spectral decomposition, polynomial functional calculus, manual block encoding, Quantum Singular Value Transformation (QSVT), and Quantum Phase Estimation (QPE) spectroscopy.

The following code defines the Pauli matrices, constructs the *Hamiltonian* explicitly, and represents it as a 4 by 4 Hermitian matrix.

In [54]:
I = np.eye(2, dtype=complex)

X = np.array(
    [[0, 1],
     [1, 0]],
    dtype=complex,
)

Y = np.array(
    [[0, -1j],
     [1j, 0]],
    dtype=complex,
)

Z = np.array(
    [[1, 0],
     [0, -1]],
    dtype=complex,
)


# Tensor product of several matrices
def tensor(*matrices: np.ndarray) -> np.ndarray:
    result = np.array([[1]], dtype=complex)
    for matrix in matrices:
        result = np.kron(result, matrix)
    return result

#Construct the Hamiltonian
H = (
    0.3 * tensor(X, X)
    - 0.2 * tensor(Z, I)
    + 0.1 * tensor(Y, Y)
)



**Spectral Analysis of the Hamiltonian**

The first step in understanding a Hamiltonian is to study its spectrum. Since the Hamiltonian is Hermitian, the spectral theorem guarantees that it has real eigenvalues and an orthonormal set of eigenvectors. The eigenvalues represent the energy levels of the system, while the eigenvectors are the corresponding energy eigenstates.

In this section, we verify that the constructed Hamiltonian is Hermitian and compute its exact eigendecomposition. These exact spectral quantities will serve as the reference throughout the remainder of the project. In later sections, we will use them to verify polynomial functional calculus, compare against manual QSVT implementations, and evaluate the accuracy of Quantum Phase Estimation.

In [55]:
print("\nHamiltonian:")
print("  H = 0.3 X⊗X - 0.2 Z⊗I + 0.1 Y⊗Y")

print("\nHamiltonian matrix H:")
print(np.round(H, 6))

print("\nHermitian check ||H - H*||:") #Check whether H is Hermitian
print(np.linalg.norm(H - H.conj().T))

print("\nSpectral norm ||H||:") #Compute the spectral norm
print(np.linalg.norm(H, 2))

eigvals_H, eigvecs_H = np.linalg.eigh(H) #Compute the eigenvalues and eigenvectors

print("\nExact eigenvalues of H:")
print(eigvals_H)

print("\nCorresponding eigenvectors:")
print(np.round(eigvecs_H,6))


Hamiltonian:
  H = 0.3 X⊗X - 0.2 Z⊗I + 0.1 Y⊗Y

Hamiltonian matrix H:
[[-0.2+0.j  0. +0.j  0. +0.j  0.2+0.j]
 [ 0. +0.j -0.2+0.j  0.4+0.j  0. +0.j]
 [ 0. +0.j  0.4+0.j  0.2+0.j  0. +0.j]
 [ 0.2+0.j  0. +0.j  0. +0.j  0.2+0.j]]

Hermitian check ||H - H*||:
0.0

Spectral norm ||H||:
0.447213595499958

Exact eigenvalues of H:
[-0.447214 -0.282843  0.282843  0.447214]

Corresponding eigenvectors:
[[-0.      +0.j  0.92388 +0.j -0.382683+0.j -0.      +0.j]
 [ 0.850651+0.j  0.      +0.j  0.      +0.j  0.525731+0.j]
 [-0.525731+0.j -0.      +0.j -0.      +0.j  0.850651+0.j]
 [-0.      +0.j -0.382683+0.j -0.92388 +0.j -0.      +0.j]]


# Part II — Spectral Functional Calculus

After computing the exact eigendecomposition of the Hamiltonian, we can study how functions act on the operator. Because the Hamiltonian is Hermitian, the spectral theorem tells us that every polynomial transformation can be performed by applying the polynomial **only to the eigenvalues**, while leaving the eigenvectors unchanged.

This idea is known as **spectral functional calculus**. If the Hamiltonian has eigenvalues λ₁, λ₂, …, λₙ, then a polynomial p(x) simply transforms them into p(λ₁), p(λ₂), …, p(λₙ). The transformed operator is then reconstructed using the original eigenvectors.

This computation provides the exact mathematical reference for the remainder of the project. Later, after manually constructing the block encoding and manually assembling the QSVT operator, we will compare their outputs against this exact result. If they agree, then the QSVT construction is correctly performing the desired polynomial transformation of the Hamiltonian spectrum.

In the following section, we evaluate the chosen polynomial on the eigenvalues, reconstruct the transformed operator, and verify the result by comparing it with the equivalent polynomial computed directly from matrix powers.

In [56]:
def polynomial_on_scalar(x: complex, coeffs: np.ndarray) -> complex:
    """
    Evaluate p(x) = c_0 + c_1 x + ... + c_d x^d.
    Coefficients are listed in increasing order.
    """
    return sum(coeff * x**power for power, coeff in enumerate(coeffs))


def polynomial_on_matrix(matrix: np.ndarray, coeffs: np.ndarray) -> np.ndarray:
    """
    Evaluate p(A) = c_0 I + c_1 A + ... + c_d A^d.
    Coefficients are listed in increasing order.
    """
    result = np.zeros_like(matrix, dtype=complex)
    identity = np.eye(matrix.shape[0], dtype=complex)

    for power, coeff in enumerate(coeffs):
        if power == 0:
            result += coeff * identity
        else:
            result += coeff * np.linalg.matrix_power(matrix, power)

    return result


def spectral_polynomial(matrix: np.ndarray, coeffs: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute p(A) using the spectral theorem.

    If A = V Lambda V*, then p(A) = V p(Lambda) V*.
    """
    eigvals, eigvecs = np.linalg.eigh(matrix)
    transformed_eigvals = np.array(
        [polynomial_on_scalar(lam, coeffs) for lam in eigvals],
        dtype=complex,
    )
    transformed_matrix = eigvecs @ np.diag(transformed_eigvals) @ eigvecs.conj().T
    return transformed_matrix, eigvals, transformed_eigvals



In [57]:
degree5_poly = np.array([0, 1.0, 0, -0.5, 0, 1 / 3], dtype=float)

H_poly_direct = polynomial_on_matrix(H, degree5_poly)
H_poly_spectral, _, H_transformed_eigvals = spectral_polynomial(H, degree5_poly)

print("\nPolynomial:")
print("  p(x) = x - x^3/2 + x^5/3")

print("\nEigenvalue transformation:")
for lam, p_lam in zip(eigvals_H, H_transformed_eigvals):
    print(f"  {lam: .6f}  --->  {np.real_if_close(p_lam): .6f}")

print("\np(H) from matrix powers:")
print(np.round(H_poly_direct, 6))

print("\np(H) from spectral decomposition:")
print(np.round(H_poly_spectral, 6))

print("\nFunctional calculus error ||p(H)_direct - p(H)_spectral||:")
print(np.linalg.norm(H_poly_direct - H_poly_spectral))


Polynomial:
  p(x) = x - x^3/2 + x^5/3

Eigenvalue transformation:
  -0.447214  --->  -0.408455
  -0.282843  --->  -0.272132
   0.282843  --->   0.272132
   0.447214  --->   0.408455

p(H) from matrix powers:
[[-0.192427+0.j  0.      +0.j  0.      +0.j  0.192427+0.j]
 [ 0.      +0.j -0.182667+0.j  0.365333+0.j  0.      +0.j]
 [ 0.      +0.j  0.365333+0.j  0.182667+0.j  0.      +0.j]
 [ 0.192427+0.j  0.      +0.j  0.      +0.j  0.192427+0.j]]

p(H) from spectral decomposition:
[[-0.192427+0.j -0.      +0.j  0.      +0.j  0.192427+0.j]
 [-0.      +0.j -0.182667+0.j  0.365333+0.j -0.      +0.j]
 [ 0.      +0.j  0.365333+0.j  0.182667+0.j -0.      +0.j]
 [ 0.192427+0.j -0.      +0.j -0.      +0.j  0.192427+0.j]]

Functional calculus error ||p(H)_direct - p(H)_spectral||:
3.6249325003368195e-16


# Part III — Manual Block Encoding of the Hamiltonian

The spectral functional calculus computation above gives the exact classical reference for \(p(H)\). The next step is to move toward the quantum algorithmic construction.

QSVT does not act directly on an arbitrary matrix. Instead, the matrix must first be embedded into a larger unitary operator. This embedding is called a **block encoding**.

In this section, we manually construct a unitary whose top-left block is the Hamiltonian \(H\). We then verify two things:

1. the larger matrix is unitary;
2. its top-left block is exactly the original Hamiltonian.

This block encoding will be the input used in the manual QSVT construction.

In [58]:
def hermitian_psd_sqrt(matrix: np.ndarray) -> np.ndarray:
    """
    Positive square root of a Hermitian positive semidefinite matrix.

    Small negative eigenvalues caused by floating point roundoff are clipped
    to zero.
    """
    eigvals, eigvecs = np.linalg.eigh(matrix)
    eigvals = np.clip(eigvals, 0.0, None)
    return eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.conj().T


def manual_block_encoding_for_hermitian_contraction(A: np.ndarray) -> np.ndarray:
    r"""
    Manually construct a block encoding of a Hermitian contraction A.

    If A = A* and ||A|| <= 1, define

        D = sqrt(I - A^2)

    and

        U_A = [[ A,  D],
               [ D, -A]].

    Then U_A is unitary and the top-left block is A.
    """
    if not np.allclose(A, A.conj().T):
        raise ValueError("This block encoding helper assumes A is Hermitian.")

    norm_A = np.linalg.norm(A, 2)
    if norm_A > 1 + 1e-12:
        raise ValueError(f"A must be a contraction. Found ||A|| = {norm_A}.")

    dim = A.shape[0]
    identity = np.eye(dim, dtype=complex)
    D = hermitian_psd_sqrt(identity - A @ A)

    return np.block([[A, D], [D, -A]])


In [59]:
U_H = manual_block_encoding_for_hermitian_contraction(H)
dim_H = H.shape[0]
dim_UH = U_H.shape[0]

print("\nBlock encoding formula:")
print("  U_H = [[H, sqrt(I-H^2)], [sqrt(I-H^2), -H]]")

print("\nTop-left block of U_H:")
print(np.round(U_H[:dim_H, :dim_H], 6))

print("\n||U_H* U_H - I||:")
print(np.linalg.norm(U_H.conj().T @ U_H - np.eye(dim_UH)))

print("\n||U_H U_H* - I||:")
print(np.linalg.norm(U_H @ U_H.conj().T - np.eye(dim_UH)))

print("\n||Top-left block - H||:")
print(np.linalg.norm(U_H[:dim_H, :dim_H] - H))



Block encoding formula:
  U_H = [[H, sqrt(I-H^2)], [sqrt(I-H^2), -H]]

Top-left block of U_H:
[[-0.2+0.j  0. +0.j  0. +0.j  0.2+0.j]
 [ 0. +0.j -0.2+0.j  0.4+0.j  0. +0.j]
 [ 0. +0.j  0.4+0.j  0.2+0.j  0. +0.j]
 [ 0.2+0.j  0. +0.j  0. +0.j  0.2+0.j]]

||U_H* U_H - I||:
2.4057875251160796e-17

||U_H U_H* - I||:
2.4057875251160796e-17

||Top-left block - H||:
0.0


# Part IV — Manual QSVT Construction

The block encoding constructed above embeds the Hamiltonian into a larger unitary matrix. We now use this unitary as the basic input for Quantum Singular Value Transformation.

The purpose of QSVT is to implement a polynomial transformation of the encoded operator. In this section, we use the same degree-5 polynomial from the spectral functional calculus section and manually assemble the QSVT operator from:

1. the block encoding of \(H\);
2. projector-controlled phase matrices;
3. alternating applications of the block encoding and its adjoint.

The only major PennyLane routine used here is the computation of QSP phase angles from the target polynomial. The actual projector matrices and the QSVT matrix product are constructed explicitly.

In [60]:
#Printing Helpers

def section(title: str) -> None:
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def subsection(title: str) -> None:
    print("\n" + "-" * 78)
    print(title)
    print("-" * 78)

In [61]:
def pcphase_matrix(phi: float, encoded_dim: int, total_dim: int) -> np.ndarray:
    """
    Matrix version of a projector-controlled phase.

    It applies exp(i phi) to the encoded subspace and exp(-i phi) to the
    orthogonal complement.
    """
    diagonal = np.empty(total_dim, dtype=complex)
    diagonal[:encoded_dim] = np.exp(1j * phi)
    diagonal[encoded_dim:] = np.exp(-1j * phi)
    return np.diag(diagonal)


def manual_qsvt_matrix(U_A: np.ndarray, projectors: list[np.ndarray], verbose: bool = False) -> np.ndarray:
    """
    Manually assemble the QSVT matrix by alternating phase projectors with
    U_A and U_A*.

    The operation sequence is

        P_0, U_A, P_1, U_A*, P_2, U_A, P_3, ...

    If operations act on states from left to right in this list, the total
    matrix is accumulated as operation @ total.
    """
    operations: list[tuple[str, np.ndarray]] = []

    for idx, projector in enumerate(projectors[:-1]):
        operations.append((f"projector phase P_{idx}", projector))

        if idx % 2 == 0:
            operations.append(("block encoding U_A", U_A))
        else:
            operations.append(("adjoint block encoding U_A*", U_A.conj().T))

    operations.append((f"projector phase P_{len(projectors) - 1}", projectors[-1]))

    if verbose:
        print("\nManual QSVT operation sequence:")
        for step, (name, _) in enumerate(operations, start=1):
            print(f"  Step {step}: apply {name}")

    total = np.eye(U_A.shape[0], dtype=complex)

    for _, operation in operations:
        total = operation @ total

    return total





In [62]:
def pennylane_qsvt_sanity_check(U_A: np.ndarray, angles: np.ndarray, encoded_dim: int) -> np.ndarray:
    """
    Use PennyLane's qml.QSVT as a sanity check only.

    This is not used as the implementation. The main QSVT implementation in
    this file is manual matrix assembly.
    """
    total_dim = U_A.shape[0]
    num_wires = int(np.log2(total_dim))

    if 2**num_wires != total_dim:
        raise ValueError("The block-encoding dimension must be a power of 2 for this sanity check.")

    wires = list(range(num_wires))
    block_encoding_operator = qml.QubitUnitary(U_A, wires=wires)

    projectors = [
        qml.PCPhase(angle, dim=encoded_dim, wires=wires)
        for angle in angles
    ]

    qsvt_operator = qml.QSVT(block_encoding_operator, projectors)
    return qml.matrix(qsvt_operator, wire_order=wires)


def run_manual_qsvt_experiment(
    A: np.ndarray,
    coeffs: np.ndarray,
    name: str,
    verbose_sequence: bool = False,
    sanity_check: bool = True,
) -> dict:
    """
    Run the full manual QSVT pipeline for one polynomial.

    Returns a dictionary containing the exact polynomial, the QSVT block,
    and error quantities.
    """
    subsection(name)

    encoded_dim = A.shape[0]
    U_A = manual_block_encoding_for_hermitian_contraction(A)
    total_dim = U_A.shape[0]

    exact = polynomial_on_matrix(A, coeffs)
    spectral, eigvals, transformed_eigvals = spectral_polynomial(A, coeffs)

    print("\nEigenvalue transformation:")
    for lam, p_lam in zip(eigvals, transformed_eigvals):
        print(f"  {lam: .6f}  --->  {np.real_if_close(p_lam): .6f}")

    print("\nExact p(A) from matrix powers:")
    print(np.round(exact, 6))

    print("\np(A) from spectral decomposition:")
    print(np.round(spectral, 6))

    spectral_error = np.linalg.norm(exact - spectral)
    print("\nSpectral functional calculus error:")
    print(spectral_error)

    # Remaining major built-in: QSP/QSVT phase-angle synthesis.
    angles = qml.poly_to_angles(coeffs, "QSVT")

    print("\nQSP/QSVT phase angles:")
    print(np.round(angles, 6))

    phase_projectors = [
        pcphase_matrix(angle, encoded_dim=encoded_dim, total_dim=total_dim)
        for angle in angles
    ]

    U_qsvt_manual = manual_qsvt_matrix(U_A, phase_projectors, verbose=verbose_sequence)

    # In this QSVT convention, the target polynomial appears in the real part
    # of the top-left block. The imaginary part is not numerical error.
    qsvt_block = U_qsvt_manual[:encoded_dim, :encoded_dim]
    manual_block = np.real(qsvt_block)
    manual_error = np.linalg.norm(manual_block - exact)

    print("\nReal part of manual QSVT top-left block:")
    print(np.round(manual_block, 6))

    print("\nManual QSVT reconstruction error:")
    print(manual_error)

    sanity_error = None
    if sanity_check:
        U_qsvt_pl = pennylane_qsvt_sanity_check(U_A, angles, encoded_dim)
        pl_block = U_qsvt_pl[:encoded_dim, :encoded_dim]
        sanity_error = np.linalg.norm(qsvt_block - pl_block)

        print("\nDifference from PennyLane qml.QSVT sanity check:")
        print(sanity_error)

    return {
        "name": name,
        "coeffs": coeffs,
        "exact": exact,
        "spectral": spectral,
        "manual_qsvt_block": manual_block,
        "spectral_error": spectral_error,
        "manual_qsvt_error": manual_error,
        "pennylane_sanity_error": sanity_error,
    }

In [63]:
result_degree5 = run_manual_qsvt_experiment(
    A=H,
    coeffs=degree5_poly,
    name="Detailed walkthrough: degree-5 polynomial transformation of H",
    verbose_sequence=True,
    sanity_check=True,
)


------------------------------------------------------------------------------
Detailed walkthrough: degree-5 polynomial transformation of H
------------------------------------------------------------------------------

Eigenvalue transformation:
  -0.447214  --->  -0.408455
  -0.282843  --->  -0.272132
   0.282843  --->   0.272132
   0.447214  --->   0.408455

Exact p(A) from matrix powers:
[[-0.192427+0.j  0.      +0.j  0.      +0.j  0.192427+0.j]
 [ 0.      +0.j -0.182667+0.j  0.365333+0.j  0.      +0.j]
 [ 0.      +0.j  0.365333+0.j  0.182667+0.j  0.      +0.j]
 [ 0.192427+0.j  0.      +0.j  0.      +0.j  0.192427+0.j]]

p(A) from spectral decomposition:
[[-0.192427+0.j -0.      +0.j  0.      +0.j  0.192427+0.j]
 [-0.      +0.j -0.182667+0.j  0.365333+0.j -0.      +0.j]
 [ 0.      +0.j  0.365333+0.j  0.182667+0.j -0.      +0.j]
 [ 0.192427+0.j -0.      +0.j -0.      +0.j  0.192427+0.j]]

Spectral functional calculus error:
3.6249325003368195e-16

QSP/QSVT phase angles:
[-5.497787

# Part V — Additional Polynomial Transformations

Part IV gave a detailed walkthrough for the degree-5 polynomial. In this section, we keep the same Hamiltonian and the same block encoding, but change the target polynomial.

This illustrates the main QSVT principle: changing the QSP phase angles changes the spectral transformation implemented by the QSVT sequence.

In [64]:
print(
    "\nPart IV already gave a detailed walkthrough for the degree-5 polynomial.\n"
    "Here we keep the same Hamiltonian and the same block encoding, but change\n"
    "the target polynomial. This demonstrates the QSVT principle: changing the\n"
    "QSP phase angles changes the spectral transformation."
)

extra_polynomials = {
    "scaled linear p(x)=0.5x": np.array([0, 0.5], dtype=float),
    "safe cubic p(x)=0.5x+0.2x^3": np.array([0, 0.5, 0, 0.2], dtype=float),
}

multi_results = []

for polynomial_name, coeffs in extra_polynomials.items():
    multi_results.append(
        run_manual_qsvt_experiment(
            A=H,
            coeffs=coeffs,
            name=polynomial_name,
            verbose_sequence=False,
            sanity_check=True,
        )
    )


Part IV already gave a detailed walkthrough for the degree-5 polynomial.
Here we keep the same Hamiltonian and the same block encoding, but change
the target polynomial. This demonstrates the QSVT principle: changing the
QSP phase angles changes the spectral transformation.

------------------------------------------------------------------------------
scaled linear p(x)=0.5x
------------------------------------------------------------------------------

Eigenvalue transformation:
  -0.447214  --->  -0.223607
  -0.282843  --->  -0.141421
   0.282843  --->   0.141421
   0.447214  --->   0.223607

Exact p(A) from matrix powers:
[[-0.1+0.j  0. +0.j  0. +0.j  0.1+0.j]
 [ 0. +0.j -0.1+0.j  0.2+0.j  0. +0.j]
 [ 0. +0.j  0.2+0.j  0.1+0.j  0. +0.j]
 [ 0.1+0.j  0. +0.j  0. +0.j  0.1+0.j]]

p(A) from spectral decomposition:
[[-0.1+0.j -0. +0.j  0. +0.j  0.1+0.j]
 [-0. +0.j -0.1+0.j  0.2+0.j -0. +0.j]
 [ 0. +0.j  0.2+0.j  0.1+0.j -0. +0.j]
 [ 0.1+0.j -0. +0.j -0. +0.j  0.1+0.j]]

Spectral functi

In [65]:
print("\nSummary of additional polynomial transformations:")
print("name                                      | spectral error | manual QSVT error | PL sanity")
print("------------------------------------------|----------------|-------------------|----------")
for result in multi_results:
    print(
        f"{result['name'][:40]:40s} | "
        f"{result['spectral_error']:.3e}   | "
        f"{result['manual_qsvt_error']:.3e}        | "
        f"{result['pennylane_sanity_error']:.3e}"
    )



Summary of additional polynomial transformations:
name                                      | spectral error | manual QSVT error | PL sanity
------------------------------------------|----------------|-------------------|----------
scaled linear p(x)=0.5x                  | 1.911e-16   | 3.743e-13        | 4.807e-17
safe cubic p(x)=0.5x+0.2x^3              | 1.892e-16   | 3.991e-13        | 1.415e-16


# Part VI — Quantum Phase Estimation (QPE) Spectroscopy

Quantum Phase Estimation (QPE) estimates the eigenvalues of a unitary operator. Since our Hamiltonian H is Hermitian rather than unitary, we first convert it into a unitary evolution operator by exponentiating a shifted and rescaled version of H.

The Hamiltonian is shifted and rescaled so that all of its eigenvalues lie between 0 and 1. This allows each energy eigenvalue to correspond to a unique phase that QPE can estimate.

In this section we:

* shift and rescale the Hamiltonian,
* construct the corresponding unitary evolution operator,
* verify the exact spectrum classically,
* run QPE on each exact eigenstate, and
* perform spectroscopy starting from a computational basis state.

The final experiment demonstrates how QPE can recover the energy spectrum from a state that is a superposition of multiple energy eigenstates.

In [66]:
def inverse_qft_matrix(M: int) -> np.ndarray:
    """Inverse QFT matrix on a counting register of dimension M."""
    y = np.arange(M).reshape((M, 1))
    k = np.arange(M).reshape((1, M))
    return np.exp(-2j * np.pi * y * k / M) / np.sqrt(M)




def qpe_distribution(U: np.ndarray, eigenstate: np.ndarray, num_counting_qubits: int) -> np.ndarray:
    """
    Manual QPE simulation.

    If U|psi> = exp(2πi theta)|psi>, QPE estimates theta.

    The state before inverse QFT is

        (1/sqrt(M)) sum_k |k> U^k |psi>.
    """
    M = 2**num_counting_qubits
    system_dim = U.shape[0]

    state = np.zeros((M, system_dim), dtype=complex)

    current = eigenstate.copy()
    for k in range(M):
        state[k, :] = current / np.sqrt(M)
        current = U @ current

    final_state = inverse_qft_matrix(M) @ state
    return np.sum(np.abs(final_state) ** 2, axis=1)

def theta_to_energy(theta: float, B: float) -> float:
    """
    Convert phase theta back to energy for the scaling

        theta = (E + B)/(2B).

    Hence

        E = 2B theta - B.
    """
    return 2 * B * theta - B

In [67]:
# QPE estimates phases theta in [0,1). To avoid ambiguity from negative
# energies, shift and rescale H:
#
#     H_scaled = (H + B I)/(2B).
#
# If E is an eigenvalue of H, then theta = (E+B)/(2B).
# QPE estimates theta, and we recover E = 2B theta - B.



B = 1.0
H_scaled = (H + B * np.eye(dim_H)) / (2 * B)
theta_exact_values = (eigvals_H + B) / (2 * B)

U_time = expm(2j * np.pi * H_scaled)

print("\nScaled eigenvalues theta = (E+B)/(2B):")
print(theta_exact_values)

print("\nUnitary check ||U*U - I||:")
print(np.linalg.norm(U_time.conj().T @ U_time - np.eye(dim_H)))

num_counting_qubits = 7
M = 2**num_counting_qubits

print(f"\nNumber of counting qubits: {num_counting_qubits}")
print(f"Phase resolution: 1/{M} = {1/M:.6f}")
print(f"Energy resolution: 2B/{M} = {2 * B / M:.6f}")

qpe_summary = []

for j, E_exact in enumerate(eigvals_H):
    psi = eigvecs_H[:, j]
    probs = qpe_distribution(U_time, psi, num_counting_qubits)

    best_y = int(np.argmax(probs))
    theta_est = best_y / M
    E_est = theta_to_energy(theta_est, B)

    theta_exact = theta_exact_values[j]
    error = abs(E_est - E_exact)

    qpe_summary.append((j, E_exact, theta_exact, best_y, theta_est, E_est, error))

    subsection(f"QPE on eigenstate {j}")
    print(f"Exact energy E       = {E_exact:.6f}")
    print(f"Exact phase theta    = {theta_exact:.6f}")
    print(f"Most likely outcome  = {best_y}")
    print(f"Estimated theta      = {theta_est:.6f}")
    print(f"Estimated energy     = {E_est:.6f}")
    print(f"Energy error         = {error:.6f}")

    print("\nTop QPE outcomes:")
    top_indices = np.argsort(probs)[-5:][::-1]
    for idx in top_indices:
        theta = idx / M
        energy = theta_to_energy(theta, B)
        print(
            f"  y={idx:3d}, theta={theta:.6f}, "
            f"E(theta)={energy:.6f}, probability={probs[idx]:.6f}"
        )



Scaled eigenvalues theta = (E+B)/(2B):
[0.276393 0.358579 0.641421 0.723607]

Unitary check ||U*U - I||:
4.897105989928208e-16

Number of counting qubits: 7
Phase resolution: 1/128 = 0.007812
Energy resolution: 2B/128 = 0.015625

------------------------------------------------------------------------------
QPE on eigenstate 0
------------------------------------------------------------------------------
Exact energy E       = -0.447214
Exact phase theta    = 0.276393
Most likely outcome  = 35
Estimated theta      = 0.273438
Estimated energy     = -0.453125
Energy error         = 0.005911

Top QPE outcomes:
  y= 35, theta=0.273438, E(theta)=-0.453125, probability=0.609411
  y= 36, theta=0.281250, E(theta)=-0.437500, probability=0.225711
  y= 34, theta=0.265625, E(theta)=-0.468750, probability=0.045930
  y= 37, theta=0.289062, E(theta)=-0.421875, probability=0.033185
  y= 33, theta=0.257812, E(theta)=-0.484375, probability=0.015438

-----------------------------------------------------

In [68]:
#QPE spectroscopy summary

print(
    "state | exact E    | exact theta | y_peak | theta_est | E_est     | error\n"
    "------|------------|-------------|--------|-----------|-----------|----------"
)

for row in qpe_summary:
    j, E_exact, theta_exact, best_y, theta_est, E_est, error = row
    print(
        f"{j:5d} | "
        f"{E_exact:10.6f} | "
        f"{theta_exact:11.6f} | "
        f"{best_y:6d} | "
        f"{theta_est:9.6f} | "
        f"{E_est:9.6f} | "
        f"{error:8.6f}"
    )

state | exact E    | exact theta | y_peak | theta_est | E_est     | error
------|------------|-------------|--------|-----------|-----------|----------
    0 |  -0.447214 |    0.276393 |     35 |  0.273438 | -0.453125 | 0.005911
    1 |  -0.282843 |    0.358579 |     46 |  0.359375 | -0.281250 | 0.001593
    2 |   0.282843 |    0.641421 |     82 |  0.640625 |  0.281250 | 0.001593
    3 |   0.447214 |    0.723607 |     93 |  0.726562 |  0.453125 | 0.005911


**Interpretation of the Results**

The small energy estimation errors are expected and do not indicate an error in the implementation. Quantum Phase Estimation estimates phases using a finite number of counting qubits. In this notebook we use a 7-qubit counting register, so the phase resolution is 1/2^7 = 1/128. Consequently, the estimated phases—and therefore the recovered energies—are rounded to the nearest representable value. The observed errors are consistent with this finite precision and demonstrate that the QPE implementation is functioning correctly.

# Part VII — QPE from a Non-Eigenstate

The previous experiment verified that Quantum Phase Estimation accurately recovers the energy of each exact eigenstate of the Hamiltonian. In practice, however, the input to QPE is rarely an eigenstate. Instead, it is typically an arbitrary quantum state that can be expressed as a superposition of energy eigenstates.

This experiment demonstrates that behavior. Rather than supplying an exact eigenvector, we initialize the system in the computational basis state |00>. Since |00> is generally a linear combination of several eigenstates, QPE no longer produces a single energy estimate. Instead, the measurement distribution exhibits multiple peaks, each corresponding to one of the Hamiltonian's eigenvalues.

The height of each peak is determined by the squared overlap between the input state and the corresponding energy eigenstate. Consequently, a single QPE circuit is able to reveal the spectral content of the input state.

This experiment illustrates the practical role of QPE as a spectroscopy algorithm: it identifies which energy levels are present in an unknown quantum state rather than merely verifying known eigenvalues.

In [69]:
ket00 = np.array([1, 0, 0, 0], dtype=complex)

overlaps = np.abs(eigvecs_H.conj().T @ ket00) ** 2



print("\nOverlap of |00⟩ with the Energy Eigenstates")
print("=" * 48)

overlap_table = pd.DataFrame({
    "State": range(len(eigvals_H)),
    "Energy": np.round(eigvals_H, 6),
    "Overlap Probability": np.round(overlaps, 6),
})

display(overlap_table)


probs_00 = qpe_distribution(U_time, ket00, num_counting_qubits)

top_indices = np.argsort(probs_00)[-8:][::-1]

qpe_table = pd.DataFrame({
    "Peak": top_indices,
    "Theta": [round(idx / M, 6) for idx in top_indices],
    "Estimated Energy": [
        round(theta_to_energy(idx / M, B), 6)
        for idx in top_indices
    ],
    "Probability": [
        round(probs_00[idx], 6)
        for idx in top_indices
    ],
})

print("\nTop QPE Outcomes from |00⟩")
print("=" * 48)

display(qpe_table)


Overlap of |00⟩ with the Energy Eigenstates


,State,Energy,Overlap Probability
0,0,-0.447214,0.000000
1,1,-0.282843,0.853553
2,2,0.282843,0.146447
3,3,0.447214,0.000000



Top QPE Outcomes from |00⟩


,Peak,Theta,Estimated Energy,Probability
0,46,0.359375,-0.281250,0.824775
1,82,0.640625,0.281250,0.141517
2,45,0.351562,-0.296875,0.010629
3,47,0.367188,-0.265625,0.007061
4,44,0.343750,-0.312500,0.002382
5,48,0.375000,-0.250000,0.001943
6,83,0.648438,0.296875,0.001832
7,81,0.632812,0.265625,0.001220


# Conclusion

In this notebook we implemented the complete spectral transformation pipeline starting from a simple two-local Hamiltonian.

Specifically, we

- constructed and analyzed a Hermitian Hamiltonian,
- verified polynomial functional calculus using the spectral theorem,
- manually built a block encoding without relying on built-in block-encoding routines,
- manually assembled the QSVT operator from projector-controlled phase matrices,
- applied several polynomial spectral transformations,
- compared every manual implementation against exact classical computations and PennyLane's built-in QSVT routine, and
- performed Quantum Phase Estimation both on exact eigenstates and on a non-eigenstate input.

The numerical experiments show that the manually constructed QSVT operator performs the intended polynomial transformation to machine precision, while QPE successfully recovers the spectrum of the Hamiltonian and reveals the spectral composition of arbitrary input states.

With the exception of `qml.poly_to_angles`, which computes the QSP phase angles, the entire QSVT pipeline is constructed manually. In particular, the block encoding, projector-controlled phase operators, QSVT circuit, and verification procedures are all implemented explicitly. PennyLane’s built-in qml.QSVT routine is used solely as an independent verification of the manually constructed implementation.